# Prediction for family

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import warnings
import joblib
from dateutil.relativedelta import relativedelta

In [2]:
# Ignoramos algunos warnings que se producen por invocar el modelo sin el nombre de las características
warnings.filterwarnings('ignore', category=UserWarning, message='.*X does not have valid feature names.*')

In [3]:
# Construcción de una función que realice el particionado completo
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

# -------------------------
# 1 -  Leemos los datos
# -------------------------

In [4]:
# Leemos el conjunto de datos
df = pd.read_csv('../../data/family-finance-family-data.csv')

In [5]:
df["year_month"] = pd.to_datetime(df["year_month"])
df["month"] = df["year_month"].dt.month
df["year"] = df["year_month"].dt.year

# Ordenamos por fecha y familia
df = df.sort_values(["family_id","year_month"]) 

# -------------------------
# 2 - Creamos features históricas
# -------------------------

In [6]:
# Ordenamos por familia y fecha
df = df.sort_values(["family_id","year_month"])

# Añadimos históricos de meses anteriores y el sexto ultimo
# Gastos
df["exp_lag1"] = df.groupby("family_id")["total_expenses"].shift(1)
df["exp_lag2"] = df.groupby("family_id")["total_expenses"].shift(2)
df["exp_lag3"] = df.groupby("family_id")["total_expenses"].shift(3)
df["exp_lag6"] = df.groupby("family_id")["total_expenses"].shift(6)

# Ingresos
df["inc_lag1"] = df.groupby("family_id")["total_income"].shift(1)
df["inc_lag2"] = df.groupby("family_id")["total_income"].shift(2)
df["inc_lag3"] = df.groupby("family_id")["total_income"].shift(3)
df["inc_lag6"] = df.groupby("family_id")["total_income"].shift(6)


# Calculamos el promedio de los tres meses anteriores
df["exp_avg3"] = df[["exp_lag1","exp_lag2","exp_lag3"]].mean(axis=1)
df["inc_avg3"] = df[["inc_lag1","inc_lag2","inc_lag3"]].mean(axis=1)

# Calculamos el promedio de los seis meses anteriores
df["exp_avg6"] = df.groupby("family_id")["total_expenses"].shift(1).rolling(6).mean().reset_index(level=0, drop=True)
df["inc_avg6"] = df.groupby("family_id")["total_income"].shift(1).rolling(6).mean().reset_index(level=0, drop=True)

# Calculamos la tendencia
df["exp_trend"] = df["exp_lag1"] - df["exp_lag3"]
df["inc_trend"] = df["inc_lag1"] - df["inc_lag3"]

# Eliminamos los primeros nulos
df = df.dropna()

# -------------------------
# 3 - Preparamos X e y
# -------------------------

In [7]:
# Features básicas
features = [
    "month",
    "exp_lag1", "exp_lag2", "exp_lag3", "exp_lag6", "exp_avg3", "exp_avg6", "exp_trend",
    "inc_lag1", "inc_lag2", "inc_lag3", "inc_lag6", "inc_avg3", "inc_avg6", "inc_trend",
]

X = df[features]

y_expenses = df["total_expenses"]
y_income = df["total_income"]

# -------------------------
# 4 - Dividimos datos en train/test
# -------------------------

In [8]:
# Dividimos los datos
X_train, X_test, y_exp_train, y_exp_test = train_test_split(
    X, y_expenses, test_size=0.2, random_state=42
)

_, _, y_inc_train, y_inc_test = train_test_split(
    X, y_income, test_size=0.2, random_state=42
)

# -------------------------
# 5 - Entrenamos Random Forest
# -------------------------

In [9]:
model_expenses = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model_income = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model_expenses.fit(X_train, y_exp_train)
model_income.fit(X_train, y_inc_train)

,n_estimators,400
,criterion,'squared_error'
,max_depth,12
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# -------------------------
# 6 - Guardamos el modelo
# -------------------------

In [10]:
# Guardamos el modelo
joblib.dump(model_expenses, "../../models/predict_family_expenses.pkl")
joblib.dump(model_income, "../../models/predict_family_income.pkl")

['../../models/predict_family_income.pkl']

In [11]:
from sklearn.metrics import mean_absolute_error

# Evalumaos
y_pred_exp = model_expenses.predict(X_test)

mae = mean_absolute_error(y_exp_test, y_pred_exp)

mean_expense = y_exp_test.mean()

print("Error medio:", mae)
print("Error relativo:", mae/mean_expense*100, "%")

Error medio: 3181.163997573824
Error relativo: 48.393631249520894 %


In [12]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

mae = mean_absolute_error(y_exp_test, y_pred_exp)
rmse = np.sqrt(mean_squared_error(y_exp_test, y_pred_exp))
r2 = r2_score(y_exp_test, y_pred_exp)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 3181.163997573824
RMSE: 4755.479312571979
R2: 0.38411371550676054


In [13]:
y_pred_inc = model_income.predict(X_test)

print("MAE inc:", mean_absolute_error(y_inc_test, y_pred_inc))

MAE inc: 2431.110732466532
